In [2]:
#!/usr/bin/env python3
import re
import sys
from io import BytesIO
from datetime import datetime

import pandas as pd
import requests

UA = {"User-Agent": "Mozilla/5.0"}
ENGLAND_PAGE = "https://www.football-data.co.uk/englandm.php"

def get_verify_arg(insecure: bool, ca_bundle: str | None):
    if insecure:
        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        return False
    if ca_bundle:
        return ca_bundle
    # default: certifi bundle
    import certifi
    return certifi.where()

def fetch_html(url: str, verify):
    r = requests.get(url, timeout=30, headers=UA, verify=verify)
    r.raise_for_status()
    return r.text

def find_e0_csv_links(html: str):
    links = re.findall(r"https?://www\.football-data\.co\.uk/mmz4281/\d{4}/E0\.csv", html)
    seen, out = set(), []
    for u in links:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def read_remote_csv(url: str, verify) -> pd.DataFrame:
    r = requests.get(url, timeout=60, headers=UA, verify=verify)
    r.raise_for_status()
    raw = r.content
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(BytesIO(raw), encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(BytesIO(raw), encoding="latin1")

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Date" in df.columns:
        def parse_date(x):
            if pd.isna(x):
                return pd.NaT
            s = str(x).strip()
            for fmt in ("%d/%m/%y", "%d/%m/%Y", "%Y-%m-%d"):
                try:
                    return datetime.strptime(s, fmt).date()
                except Exception:
                    pass
            return pd.NaT
        df["DateISO"] = df["Date"].apply(parse_date)

    rename_map = {"HomeTeam":"home_team","AwayTeam":"away_team","FTHG":"home_goals","FTAG":"away_goals","FTR":"ft_result","Div":"division"}
    for k,v in rename_map.items():
        if k in df.columns:
            df = df.rename(columns={k:v})
    return df

def main(out_csv: str, insecure: bool = False, ca_bundle: str | None = None):
    verify = get_verify_arg(insecure=insecure, ca_bundle=ca_bundle)

    html = fetch_html(ENGLAND_PAGE, verify=verify)
    links = find_e0_csv_links(html)
    if not links:
        raise RuntimeError("No E0.csv links found. Site format may have changed.")

    frames = []
    for u in links:
        print("Downloading:", u)
        df = read_remote_csv(u, verify=verify)
        df = normalize_columns(df)
        m = re.search(r"/(\d{4})/E0\.csv$", u)
        if m:
            df["season_code"] = m.group(1)
        df["source_url"] = u
        frames.append(df)

    all_df = pd.concat(frames, ignore_index=True)

    key_cols = [c for c in ["Date", "home_team", "away_team"] if c in all_df.columns]
    if len(key_cols) == 3:
        all_df = all_df.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

    if "DateISO" in all_df.columns and "home_team" in all_df.columns and "away_team" in all_df.columns:
        all_df = all_df.sort_values(["DateISO","home_team","away_team"], na_position="last").reset_index(drop=True)

    all_df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv} | rows={len(all_df)} cols={len(all_df.columns)}")

if __name__ == "__main__":
    # usage:
    #   python epl_dump.py output.csv
    #   python epl_dump.py output.csv --insecure
    #   python epl_dump.py output.csv --ca_bundle /path/to/ca.pem
    out = sys.argv[1] if len(sys.argv) > 1 else "epl_all.csv"
    insecure = ("--insecure" in sys.argv)
    ca_bundle = None
    if "--ca_bundle" in sys.argv:
        i = sys.argv.index("--ca_bundle")
        ca_bundle = sys.argv[i+1]
    main(out, insecure=insecure, ca_bundle=ca_bundle)

SSLError: HTTPSConnectionPool(host='www.football-data.co.uk', port=443): Max retries exceeded with url: /englandm.php (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:997)')))

In [ ]:
#!/usr/bin/env python3
"""
EPL Transformer betting model (Home/Draw/Away)

Input: a match-by-match CSV with at least:
  date, home_team, away_team, home_goals, away_goals
Optional (recommended) numeric columns:
  home_xg, away_xg, home_shots, away_shots, ... etc.

Model:
  - Builds sequences for each match:
      * last N matches of home team (as a token sequence)
      * last N matches of away team (as a token sequence)
      * context features for the current match (difference features, home indicator, rest days, etc.)
  - Transformer encoder over tokens, then pooled + context => 3-way classification.

Usage:
  python epl_transformer.py --csv data/epl.csv --date_col date \
      --home_team_col home_team --away_team_col away_team \
      --home_goals_col home_goals --away_goals_col away_goals \
      --val_start_date 2025-08-01 --seq_len 8 --epochs 20

Notes:
  - This is a baseline transformer for tabular sequential "form" modeling.
  - It is NOT a guaranteed profitable strategy; you'll still need odds + EV logic to backtest betting.
"""

import argparse
import math
import os
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ---------------------------
# Utilities
# ---------------------------

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def compute_1x2_label(home_goals: int, away_goals: int) -> int:
    # 0=Home, 1=Draw, 2=Away
    if home_goals > away_goals:
        return 0
    if home_goals == away_goals:
        return 1
    return 2


# ---------------------------
# Feature Engineering
# ---------------------------

@dataclass
class Cols:
    date: str
    home_team: str
    away_team: str
    home_goals: str
    away_goals: str

def infer_numeric_feature_cols(df: pd.DataFrame, cols: Cols) -> List[str]:
    """Pick numeric columns that are NOT target/ID columns."""
    exclude = {
        cols.date, cols.home_team, cols.away_team,
        cols.home_goals, cols.away_goals
    }
    # keep numeric columns
    num_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            num_cols.append(c)
    return num_cols

def add_basic_team_features(df: pd.DataFrame, cols: Cols) -> pd.DataFrame:
    """
    Create some universal numeric features that don't require external data.
    These help even when you only have goals.
    """
    out = df.copy()

    # goals for/against in current match (known after match; used only for history tokens)
    out["home_gf"] = out[cols.home_goals].astype(float)
    out["home_ga"] = out[cols.away_goals].astype(float)
    out["away_gf"] = out[cols.away_goals].astype(float)
    out["away_ga"] = out[cols.home_goals].astype(float)

    # outcome points in current match (used only for history)
    out["home_pts"] = np.where(out[cols.home_goals] > out[cols.away_goals], 3,
                       np.where(out[cols.home_goals] == out[cols.away_goals], 1, 0)).astype(float)
    out["away_pts"] = np.where(out[cols.away_goals] > out[cols.home_goals], 3,
                       np.where(out[cols.away_goals] == out[cols.home_goals], 1, 0)).astype(float)

    # goal difference
    out["home_gd"] = out["home_gf"] - out["home_ga"]
    out["away_gd"] = out["away_gf"] - out["away_ga"]

    return out

def build_team_match_table(df: pd.DataFrame, cols: Cols, extra_numeric_cols: List[str]) -> pd.DataFrame:
    """
    Build a 'long' table with one row per team per match.
    Contains team perspective features for the match (to be used in history sequences).
    """
    # For "team perspective", convert home/away columns to unified schema.
    base_cols = [cols.date, cols.home_team, cols.away_team, cols.home_goals, cols.away_goals]
    keep = list(dict.fromkeys(base_cols + extra_numeric_cols + [
        "home_gf", "home_ga", "home_pts", "home_gd",
        "away_gf", "away_ga", "away_pts", "away_gd",
    ]))

    d = df[keep].copy()

    # Home perspective rows
    home_rows = pd.DataFrame({
        "date": d[cols.date],
        "team": d[cols.home_team],
        "is_home": 1.0,
        "gf": d["home_gf"],
        "ga": d["home_ga"],
        "gd": d["home_gd"],
        "pts": d["home_pts"],
        "opp": d[cols.away_team],
    })

    # Away perspective rows
    away_rows = pd.DataFrame({
        "date": d[cols.date],
        "team": d[cols.away_team],
        "is_home": 0.0,
        "gf": d["away_gf"],
        "ga": d["away_ga"],
        "gd": d["away_gd"],
        "pts": d["away_pts"],
        "opp": d[cols.home_team],
    })

    # Optional: fold extra numeric columns into perspective.
    # If your CSV has columns like home_xg / away_xg, include them in extra_numeric_cols.
    # We will try to map them if they follow a home_/away_ prefix convention.
    for c in extra_numeric_cols:
        lc = c.lower()
        if lc.startswith("home_"):
            feat = lc.replace("home_", "")
            home_rows[feat] = df[c].astype(float).values
        elif lc.startswith("away_"):
            feat = lc.replace("away_", "")
            away_rows[feat] = df[c].astype(float).values
        else:
            # If it doesn't have home_/away_ prefix, it's ambiguous; ignore in perspective.
            pass

    long_df = pd.concat([home_rows, away_rows], axis=0, ignore_index=True)
    long_df = long_df.sort_values(["team", "date"]).reset_index(drop=True)
    return long_df


# ---------------------------
# Dataset
# ---------------------------

class MatchSequenceDataset(Dataset):
    """
    Each item corresponds to one match (row in df_matches).
    It returns:
      - token_seq: (T, F) where T=2*seq_len (home hist tokens + away hist tokens)
      - context: (C,) features for current match (home vs away recent averages etc.)
      - label: int in {0,1,2}
      - mask: (T,) indicating which tokens are real vs padding
    """

    def __init__(
        self,
        df_matches: pd.DataFrame,
        df_team_long: pd.DataFrame,
        cols: Cols,
        seq_len: int = 8,
        min_history: int = 3,
        team_feat_cols: Optional[List[str]] = None,
    ):
        self.df = df_matches.reset_index(drop=True)
        self.cols = cols
        self.seq_len = seq_len
        self.min_history = min_history

        # choose team-level history feature columns
        base = ["is_home", "gf", "ga", "gd", "pts"]
        # also include any other numeric columns present in long df besides ids
        if team_feat_cols is None:
            team_feat_cols = [c for c in df_team_long.columns
                              if c not in {"date", "team", "opp"} and pd.api.types.is_numeric_dtype(df_team_long[c])]
        # ensure base present
        for b in base:
            if b not in team_feat_cols:
                team_feat_cols = [b] + team_feat_cols

        # de-duplicate preserving order
        seen = set()
        team_feat_cols = [c for c in team_feat_cols if not (c in seen or seen.add(c))]

        self.team_feat_cols = team_feat_cols
        self.F = len(self.team_feat_cols)

        # index long table by team for fast lookup
        self.long = df_team_long.copy()
        # Keep as numpy for speed
        self.long["date"] = pd.to_datetime(self.long["date"])
        self.long = self.long.sort_values(["team", "date"]).reset_index(drop=True)

        self.team_to_idx = {}
        self.team_rows = {}
        for team, g in self.long.groupby("team", sort=False):
            self.team_to_idx[team] = len(self.team_to_idx)
            self.team_rows[team] = g.reset_index(drop=True)

        # Precompute valid match indices (enough history for both teams)
        self.df["date"] = pd.to_datetime(self.df[cols.date])
        self.df = self.df.sort_values(cols.date).reset_index(drop=True)

        self.valid_idx = []
        for i, row in self.df.iterrows():
            ht = row[cols.home_team]
            at = row[cols.away_team]
            d = row["date"]
            if ht not in self.team_rows or at not in self.team_rows:
                continue
            if self._count_history_before(ht, d) >= self.min_history and self._count_history_before(at, d) >= self.min_history:
                self.valid_idx.append(i)

    def _count_history_before(self, team: str, date: pd.Timestamp) -> int:
        g = self.team_rows[team]
        # binary search
        return int((g["date"] < date).sum())

    def _get_history_tokens(self, team: str, date: pd.Timestamp) -> Tuple[np.ndarray, np.ndarray]:
        """
        Return (tokens, mask) for last seq_len games BEFORE date.
        tokens shape: (seq_len, F)
        mask shape: (seq_len,) 1 for real token, 0 for pad
        """
        g = self.team_rows[team]
        hist = g[g["date"] < date]
        hist = hist.tail(self.seq_len)
        tokens = np.zeros((self.seq_len, self.F), dtype=np.float32)
        mask = np.zeros((self.seq_len,), dtype=np.float32)

        if len(hist) > 0:
            vals = hist[self.team_feat_cols].astype(np.float32).values
            # right-align tokens (pad on the left)
            tokens[-len(vals):, :] = vals
            mask[-len(vals):] = 1.0
        return tokens, mask

    def _context_features(self, ht: str, at: str, date: pd.Timestamp) -> np.ndarray:
        """
        Simple context: difference of recent averages over last seq_len games.
        """
        h_tok, h_mask = self._get_history_tokens(ht, date)
        a_tok, a_mask = self._get_history_tokens(at, date)

        def masked_mean(x, m):
            denom = m.sum() + 1e-6
            return (x * m[:, None]).sum(axis=0) / denom

        h_mean = masked_mean(h_tok, h_mask)
        a_mean = masked_mean(a_tok, a_mask)

        # context = [h_mean, a_mean, h_mean - a_mean]
        ctx = np.concatenate([h_mean, a_mean, (h_mean - a_mean)], axis=0).astype(np.float32)
        return ctx

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, idx: int):
        i = self.valid_idx[idx]
        row = self.df.iloc[i]
        date = row["date"]
        ht = row[self.cols.home_team]
        at = row[self.cols.away_team]

        # tokens: home hist then away hist
        h_tok, h_mask = self._get_history_tokens(ht, date)
        a_tok, a_mask = self._get_history_tokens(at, date)

        tokens = np.concatenate([h_tok, a_tok], axis=0)  # (2*seq_len, F)
        mask = np.concatenate([h_mask, a_mask], axis=0)  # (2*seq_len,)

        ctx = self._context_features(ht, at, date)  # (3*F,)

        label = compute_1x2_label(int(row[self.cols.home_goals]), int(row[self.cols.away_goals]))

        return {
            "tokens": torch.from_numpy(tokens),
            "mask": torch.from_numpy(mask),
            "context": torch.from_numpy(ctx),
            "label": torch.tensor(label, dtype=torch.long),
        }


# ---------------------------
# Model
# ---------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class MatchTransformer(nn.Module):
    def __init__(
        self,
        token_dim: int,
        ctx_dim: int,
        d_model: int = 128,
        nhead: int = 8,
        num_layers: int = 3,
        dim_ff: int = 256,
        dropout: float = 0.15,
        num_classes: int = 3,
        max_len: int = 64,
    ):
        super().__init__()
        self.token_proj = nn.Linear(token_dim, d_model)
        self.pos = PositionalEncoding(d_model, max_len=max_len)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.ctx_mlp = nn.Sequential(
            nn.Linear(ctx_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
        )

        self.head = nn.Sequential(
            nn.Linear(d_model + d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, tokens: torch.Tensor, mask: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        """
        tokens: (B, T, F)
        mask:   (B, T) 1 for real tokens, 0 for padding
        context:(B, C)
        """
        x = self.token_proj(tokens)  # (B,T,D)
        x = self.pos(x)

        # Transformer wants True for positions to ignore if using src_key_padding_mask
        key_padding_mask = (mask < 0.5)  # (B,T) bool
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)  # (B,T,D)

        # masked mean pool
        m = mask.unsqueeze(-1)  # (B,T,1)
        pooled = (x * m).sum(dim=1) / (m.sum(dim=1) + 1e-6)  # (B,D)

        ctx = self.ctx_mlp(context)  # (B,D)

        out = self.head(torch.cat([pooled, ctx], dim=-1))
        return out


# ---------------------------
# Train / Eval
# ---------------------------

@torch.no_grad()
def evaluate(model, loader, device) -> Dict[str, float]:
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    ce = nn.CrossEntropyLoss()

    # For calibration-ish: collect probs for Brier
    probs_all = []
    y_all = []

    for batch in loader:
        tokens = batch["tokens"].to(device)
        mask = batch["mask"].to(device)
        context = batch["context"].to(device)
        y = batch["label"].to(device)

        logits = model(tokens, mask, context)
        loss = ce(logits, y)
        loss_sum += float(loss.item()) * y.size(0)

        pred = logits.argmax(dim=-1)
        correct += int((pred == y).sum().item())
        total += y.size(0)

        p = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        probs_all.append(p)
        y_all.append(y.detach().cpu().numpy())

    probs_all = np.concatenate(probs_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    # Brier score (multi-class)
    y_onehot = np.eye(3)[y_all]
    brier = np.mean(np.sum((probs_all - y_onehot) ** 2, axis=1))

    return {
        "loss": loss_sum / max(total, 1),
        "acc": correct / max(total, 1),
        "brier": float(brier),
        "n": float(total),
    }

def train(
    model,
    train_loader,
    val_loader,
    device,
    lr: float = 2e-4,
    weight_decay: float = 1e-3,
    epochs: int = 20,
    grad_clip: float = 1.0,
):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    ce = nn.CrossEntropyLoss()

    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        run_loss = 0.0
        n = 0

        for batch in train_loader:
            tokens = batch["tokens"].to(device)
            mask = batch["mask"].to(device)
            context = batch["context"].to(device)
            y = batch["label"].to(device)

            logits = model(tokens, mask, context)
            loss = ce(logits, y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            run_loss += float(loss.item()) * y.size(0)
            n += y.size(0)

        train_loss = run_loss / max(n, 1)
        val_metrics = evaluate(model, val_loader, device)

        print(
            f"Epoch {ep:02d} | train_loss={train_loss:.4f} "
            f"| val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.3f} "
            f"val_brier={val_metrics['brier']:.4f} (n={int(val_metrics['n'])})"
        )

        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


# ---------------------------
# Main
# ---------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True, help="Path to matches CSV")
    ap.add_argument("--date_col", default="date")
    ap.add_argument("--home_team_col", default="home_team")
    ap.add_argument("--away_team_col", default="away_team")
    ap.add_argument("--home_goals_col", default="home_goals")
    ap.add_argument("--away_goals_col", default="away_goals")

    ap.add_argument("--val_start_date", required=True, help="YYYY-MM-DD start date for validation split")
    ap.add_argument("--seq_len", type=int, default=8)
    ap.add_argument("--min_history", type=int, default=3)

    ap.add_argument("--batch_size", type=int, default=128)
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--weight_decay", type=float, default=1e-3)

    ap.add_argument("--d_model", type=int, default=128)
    ap.add_argument("--nhead", type=int, default=8)
    ap.add_argument("--num_layers", type=int, default=3)
    ap.add_argument("--dim_ff", type=int, default=256)
    ap.add_argument("--dropout", type=float, default=0.15)

    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--save", default="model.pt")
    args = ap.parse_args()

    set_seed(args.seed)

    cols = Cols(
        date=args.date_col,
        home_team=args.home_team_col,
        away_team=args.away_team_col,
        home_goals=args.home_goals_col,
        away_goals=args.away_goals_col,
    )

    df = pd.read_csv(args.csv)
    if cols.date not in df.columns:
        raise ValueError(f"Missing date column: {cols.date}")
    for c in [cols.home_team, cols.away_team, cols.home_goals, cols.away_goals]:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    df[cols.date] = pd.to_datetime(df[cols.date])
    df = df.sort_values(cols.date).reset_index(drop=True)

    # Add built-in team features (gf/ga/pts/gd)
    df = add_basic_team_features(df, cols)

    # Detect optional numeric columns (e.g., home_xg, away_xg, home_shots, away_shots)
    extra_numeric = infer_numeric_feature_cols(df, cols)

    # Build long table for history sequences
    df_long = build_team_match_table(df, cols, extra_numeric_cols=extra_numeric)

    # Split by time (train before val_start_date)
    val_start = pd.to_datetime(args.val_start_date)
    df_train = df[df[cols.date] < val_start].copy()
    df_val = df[df[cols.date] >= val_start].copy()

    if len(df_train) < 200:
        print("Warning: very small training set. Consider earlier seasons.")
    if len(df_val) < 50:
        print("Warning: very small validation set. Consider adjusting val_start_date.")

    # Dataset
    train_ds = MatchSequenceDataset(
        df_matches=df_train,
        df_team_long=df_long,
        cols=cols,
        seq_len=args.seq_len,
        min_history=args.min_history,
        team_feat_cols=None,
    )
    val_ds = MatchSequenceDataset(
        df_matches=df_val,
        df_team_long=df_long,
        cols=cols,
        seq_len=args.seq_len,
        min_history=args.min_history,
        team_feat_cols=train_ds.team_feat_cols,  # keep consistent
    )

    print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
    if len(train_ds) == 0 or len(val_ds) == 0:
        raise RuntimeError("Not enough samples after min_history filter. Lower --min_history or include more seasons.")

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)

    token_dim = train_ds.F
    ctx_dim = 3 * train_ds.F
    max_len = 2 * args.seq_len

    model = MatchTransformer(
        token_dim=token_dim,
        ctx_dim=ctx_dim,
        d_model=args.d_model,
        nhead=args.nhead,
        num_layers=args.num_layers,
        dim_ff=args.dim_ff,
        dropout=args.dropout,
        max_len=max_len,
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")

    model = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        lr=args.lr,
        weight_decay=args.weight_decay,
        epochs=args.epochs,
    )

    torch.save(
        {
            "state_dict": model.state_dict(),
            "team_feat_cols": train_ds.team_feat_cols,
            "seq_len": args.seq_len,
            "cols": cols.__dict__,
        },
        args.save,
    )
    print(f"Saved: {args.save}")

    final_metrics = evaluate(model, val_loader, device)
    print("Final val:", final_metrics)


if __name__ == "__main__":
    main()